# 06: Drift detection (ADR-008)

Question: *has the data the model sees in production moved away from the data it was trained on?*

- **Reference** = the training split (X_train). **Current** = a window of incoming data.
- Per feature: KS test (numeric) or chi-square (categorical) **and** PSI. A feature is *drifted* only if PSI >= 0.2 **and** p < 0.05.
- Dataset verdict: `drift` when >= 30% of features drifted; `insufficient_data` below 100 rows.
- Drift is **not** accuracy loss. Section 5 shows the difference using labels we happen to have here; in production labels arrive late.

Scenarios: (1) the untouched test set, which must show **no drift**; (2) the test set resampled with increasing pressure toward short-tenure, month-to-month, fiber customers, which should drift once the shift is large enough. Nothing here is retrained; Phase 8 acts on the verdict.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml
EVIDENTLY_SPEC = "evidently>=0.7"   # HTML report only; the verdict comes from ai_experiment_lab.monitoring

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

from ai_experiment_lab.data import load_csv
from ai_experiment_lab.evaluation import classification_metrics
from ai_experiment_lab.monitoring import detect_drift
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.registry.mlflow_registry import load_by_alias
from ai_experiment_lab.tracking import setup_tracking
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
TH = load_config(f"{REPO_DIR}/configs/monitoring.yaml")["drift"]
d, sp, seed = cfg["dataset"], cfg["split"], cfg["project"]["random_seed"]
setup_tracking(cfg["tracking"]["experiment_name"])

df, info = load_csv(Path(get_data_dir()) / d["file"])
split = split_data(df, d["target"], d["positive_label"], d["id_columns"], sp["test_size"], seed, sp["stratify"])
pipe = load_by_alias("telco-churn-classifier")           # the champion (ADR-005)
REF, TEST = split.X_train, split.X_test
ref_scores = pipe.predict_proba(REF)[:, 1]
print("thresholds:", TH)
print("reference rows:", len(REF), "| test rows:", len(TEST))

## 1. Scenario 1: untouched test set (expect no drift)

In [ ]:
rep1 = detect_drift(REF, TEST, **TH, reference_scores=ref_scores,
                    current_scores=pipe.predict_proba(TEST)[:, 1])
print(rep1.summary())
rep1.to_frame().sort_values("psi", ascending=False).round(4)

## 2. Scenario 2: shifted data, with increasing strength

Rows are resampled from the test set with weight `1 + strength` for customers who are month-to-month, on fiber, or have tenure under 12 months (the profile that churns most). Strength 0 is a plain bootstrap of the test set (a control: should not drift).

In [ ]:
def shifted_sample(X, strength, n=1000, seed=0):
    risky = ((X["Contract"] == "Month-to-month").astype(int)
             + (X["InternetService"] == "Fiber optic").astype(int)
             + (X["tenure"] < 12).astype(int))
    w = 1 + strength * risky
    return X.sample(n, replace=True, weights=w, random_state=seed)

rows, reports = [], {}
for strength in [0, 1, 3, 10, 30, 100, 1000, 10000]:
    cur = shifted_sample(TEST, strength)
    rep = detect_drift(REF, cur, **TH, reference_scores=ref_scores,
                       current_scores=pipe.predict_proba(cur)[:, 1])
    reports[strength] = (cur, rep)
    strict = detect_drift(REF, cur, **{**TH, "psi_threshold": 0.2})     # the conventional 0.2, for comparison
    rows.append({"strength": strength, "verdict": rep.verdict,
                 "drifted_features": len(rep.drifted_features), "of": len(rep.features),
                 "share": round(rep.drifted_share, 2),
                 "n@psi0.2": len(strict.drifted_features), "verdict@psi0.2": strict.verdict,
                 "prediction_drift": rep.prediction.drifted,
                 "mean_score": round(float(pipe.predict_proba(cur)[:, 1].mean()), 3)})
sens = pd.DataFrame(rows)
sens

The configured PSI threshold is in `TH`; the `@psi0.2` columns show the conventional 0.2 for comparison. Read the table as a sensitivity curve: where the verdict flips from `no_drift` to `drift` is how large a shift the thresholds need before they fire. If the largest strength still says `no_drift`, the thresholds are too lax for this data and ADR-008 must be revisited; if strength 0 says `drift`, they are too tight.

In [ ]:
STRENGTH = next((r["strength"] for r in rows if r["verdict"] == "drift"), 10000)
cur_shift, rep2 = reports[STRENGTH]
print("Chosen scenario strength:", STRENGTH)
print(rep2.summary())
rep2.to_frame().sort_values("psi", ascending=False).round(4)

## 3. Insufficient data guard
A handful of rows can look dramatically different by chance, so the verdict must not be `drift` on a tiny window.

In [ ]:
tiny = detect_drift(REF, cur_shift.head(30), **TH)
print(tiny.verdict, "| rows:", tiny.n_current)
assert tiny.verdict == "insufficient_data"

## 4. Save the reports (JSON) to Drive
The JSON is what Phase 8's retraining trigger reads.

In [ ]:
OUT = Path(DRIVE_ROOT) / "reports" / "drift"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "drift_no_shift.json").write_text(json.dumps(rep1.to_dict(), indent=2, default=str))
(OUT / "drift_shifted.json").write_text(json.dumps(rep2.to_dict(), indent=2, default=str))
sens.to_csv(OUT / "sensitivity.csv", index=False)
print("saved to", OUT)

## 5. Drift versus performance (labels are available here only because this is a simulation)
Drift is a warning about *inputs*. Whether the model got worse has to be measured with true labels. Here we have them, so we can check whether the drift verdict and the accuracy loss agree. In production the labels arrive weeks later (customers churn after the fact), so drift is the early signal and performance is the delayed confirmation.

In [ ]:
def perf(X, y):
    return classification_metrics(y, pipe.predict(X), pipe.predict_proba(X)[:, 1])

base = perf(TEST, split.y_test)
cur_y = split.y_test.loc[cur_shift.index]        # labels of the resampled rows (index is kept by .sample)
shifted = perf(cur_shift, cur_y)
print(f"{'':14}{'f1':>8}{'roc_auc':>10}{'precision':>11}{'recall':>8}   churn rate")
print(f"{'test set':14}{base['f1']:8.3f}{base['roc_auc']:10.3f}{base['precision']:11.3f}{base['recall']:8.3f}   {split.y_test.mean():.3f}")
print(f"{'shifted':14}{shifted['f1']:8.3f}{shifted['roc_auc']:10.3f}{shifted['precision']:11.3f}{shifted['recall']:8.3f}   {cur_y.mean():.3f}")

Interpretation guide: a shifted population has a different churn *rate*, so F1 and precision move even when the model's ranking quality (ROC AUC) does not. A drift alarm with stable ROC AUC means "inputs changed, ranking still fine; watch it". A drift alarm with falling ROC AUC means retrain. The notebook output decides which case this is; record it in the Phase 7 report.

## 6. Evidently report (HTML) for people to read
The library's own implementation is used only to produce a readable report; its verdict is cross-checked against ours. Evidently's API changed between 0.4 and 0.7, so both are handled.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", EVIDENTLY_SPEC], check=False)
import evidently
print("evidently", evidently.__version__)

html_path = OUT / "evidently_shifted.html"
ev_share = None
try:                                                    # evidently >= 0.7
    from evidently import Report
    from evidently.presets import DataDriftPreset
    ev = Report([DataDriftPreset()]).run(cur_shift.reset_index(drop=True), REF.reset_index(drop=True))
    ev.save_html(str(html_path))
    res = ev.dict()
    print("keys:", list(res)[:6])
except ImportError:                                     # evidently 0.4.x
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    ev = Report(metrics=[DataDriftPreset()])
    ev.run(reference_data=REF.reset_index(drop=True), current_data=cur_shift.reset_index(drop=True))
    ev.save_html(str(html_path))
print("saved", html_path)
for m in (ev.dict().get("metrics", []) if hasattr(ev, "dict") else []):
    if "DriftedColumnsCount" in str(m.get("metric_id", m.get("metric", ""))):
        print(m)

Download `evidently_shifted.html` from Drive (`reports/drift/`) and open it in a browser for the write-up. Our module and Evidently use different tests and thresholds, so they can disagree on borderline columns; large shifts should agree.

## What to paste back
The output of cells 3, 4 (scenario 1), 5 (the sensitivity table), 6 (shifted summary), 7 (insufficient data), and 9 (performance table), plus the "evidently ... saved" line.